# 교재원문 배치 리플로우 노트북

> Google Drive 마운트 → 1,239개 마크다운 파일 일괄 전처리 + 구조화

## 파이프라인 구조
1. **Phase 1**: 정규식 기반 노이즈 제거 (reflow_batch.py 로직) — **Colab**
2. **Phase 2**: 규칙 기반 구조화 (볼드, 번호 리스트화, 조문번호 볼드) — **Colab**
3. **Phase 3**: LLM 패스 (헤딩 위계, 학설 표, 판례 콜아웃 등) — **Claude Code 에이전트 병렬 처리** (이 노트북 아님)

Phase 1-2만 이 노트북에서 실행. Phase 3은 Claude Code 세션에서 에이전트로 처리.

## 0. 환경 설정

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive'  # H:\내 드라이브
TEXTBOOK_DIR = os.path.join(BASE_DIR, 'sync/_교재원문')
print(f'교재원문 디렉토리: {os.path.exists(TEXTBOOK_DIR)}')
print(f'하위 과목: {os.listdir(TEXTBOOK_DIR)}')

In [ ]:
# Phase 1+2는 추가 패키지 불필요 (표준 라이브러리만 사용)
# Gemini API 사용 시에만: !pip install -q google-generativeai
print("추가 패키지 불필요")

In [ ]:
import re
import json
import shutil
import time
from pathlib import Path
from datetime import datetime
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

# 리포트 저장 경로
REPORT_DIR = os.path.join(BASE_DIR, '.agent/state')
os.makedirs(REPORT_DIR, exist_ok=True)

# 백업 설정
BACKUP_ENABLED = True
BACKUP_SUFFIX = '_backup_colab'

## 1. Phase 1 — 정규식 노이즈 제거

reflow_batch.py와 동일한 로직. 이미 적용된 파일도 멱등하게 재실행 가능.

In [ ]:
# ═══════════════════════════════════════
# Phase 1: 정규식 패턴 정의
# ═══════════════════════════════════════

# 1. 페이지 헤더/푸터
PAGE_HEADER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s*[|I]\s*Part\s*\d+\.\s*.+$', re.MULTILINE),
    re.compile(r'^\s*\d{1,3}\s+.{5,60}\s*[|I]\s*\d{2,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*\d{1,3}\s*$', re.MULTILINE),
    re.compile(r'^\s*LawSchool\s+Civil\s+Act\s*$', re.MULTILINE),
    re.compile(r'^\s*재산법\s*$', re.MULTILINE),
    re.compile(r'^\s*\(돈절\]\s*$', re.MULTILINE),
    re.compile(r'^\s*제\d+\s*편\s*$', re.MULTILINE),
]
PAGE_COMMENT = re.compile(r'<!--\s*page\s+\d+\s*-->', re.IGNORECASE)

# 2. 가짜 헤딩
FALSE_HEADING = re.compile(r'^(#{1,6})\s+I\.\s+(.+)$', re.MULTILINE)

# 3. 줄 노이즈
TRAILING_PIPE = re.compile(r'\s*[|I]\s*$', re.MULTILINE)
LEADING_I = re.compile(r'^[Ii]\s+(?=[가-힣])', re.MULTILINE)

# 4. 판례번호 수정
CASE_FIXES = [
    (re.compile(r'(\d{2,4})Q(\d{3,6})'), r'\g<1>다\g<2>'),
    (re.compile(r'(\d{2,4})水(\d{3,6})'), r'\g<1>다\g<2>'),
    (re.compile(r'(\d{2,4})C\|(\d{3,6})'), r'\g<1>다\g<2>'),
    (re.compile(r'(\d{2,4})CI(\d{3,6})'), r'\g<1>다\g<2>'),
    (re.compile(r'(\d{2,4})C\^(\d{3,6})'), r'\g<1>다\g<2>'),
    (re.compile(r'(\d{2,4})P(\d{3,6})'), r'\g<1>다\g<2>'),
    (re.compile(r'대팬'), '대판'),
    (re.compile(r'대[®©]\*?\)?'), '대판'),
    (re.compile(r'대판\(:?M\)'), '대판(전합)'),
    (re.compile(r'대판\(M\)'), '대판(전합)'),
    (re.compile(r'\(全合\)'), '(전합)'),
    (re.compile(r'대EKM\)'), '대판(전합)'),
    (re.compile(r'대판7(\d{4})'), r'대판 \g<1>'),
    (re.compile(r'법정지싱권'), '법정지상권'),
    (re.compile(r'법정자상권'), '법정지상권'),
    (re.compile(r'피딤보'), '피담보'),
    (re.compile(r'(?<=[가-힣])乂'), 'X'),
    (re.compile(r'꽌'), '만'),
    (re.compile(r'으\|사'), '의사'),
    (re.compile(r'刀\|등기'), '가등기'),
    (re.compile(r'재산밭'), '재산법'),
    (re.compile(r't대판'), '(대판'),
    (re.compile(r'!대판'), '(대판'),
    (re.compile(r'＜대판'), '(대판'),
]

# 5. OCR 아티팩트
OCR_FIXES = [
    (re.compile(r'｛'), '('),
    (re.compile(r'｝'), ')'),
    (re.compile(r'〈(?!대판)'), '('),
    (re.compile(r'〉'), ')'),
    (re.compile(r'；'), ';'),
    (re.compile(r'=■'), ''),
    (re.compile(r'=＞'), '→'),
    (re.compile(r'■=＞'), '→'),
    (re.compile(r'—>'), '→'),
    (re.compile(r'^。', re.MULTILINE), ''),
]

# 6. 한자 → 한글
HANJA = [('甲','갑'),('乙','을'),('丙','병'),('丁','정'),('戊','무'),
         ('己','기'),('庚','경'),('辛','신'),('壬','임'),('癸','계'),
         ('條','조'),('項','항'),('號','호')]

# 7. 띄어쓰기
SPACING = [
    (re.compile(r'할수있'), '할 수 있'), (re.compile(r'할수없'), '할 수 없'),
    (re.compile(r'볼수있'), '볼 수 있'), (re.compile(r'볼수없'), '볼 수 없'),
    (re.compile(r'볼수는'), '볼 수는'), (re.compile(r'것으로볼'), '것으로 볼'),
    (re.compile(r'하는것'), '하는 것'), (re.compile(r'되는것'), '되는 것'),
    (re.compile(r'있는것'), '있는 것'), (re.compile(r'없는것'), '없는 것'),
    (re.compile(r'될수'), '될 수'), (re.compile(r'있을수'), '있을 수'),
    (re.compile(r'그건물'), '그 건물'), (re.compile(r'그토지'), '그 토지'),
    (re.compile(r'그부지'), '그 부지'), (re.compile(r'그지상'), '그 지상'),
    (re.compile(r'그후'), '그 후'), (re.compile(r'그효력'), '그 효력'),
    (re.compile(r'그지분'), '그 지분'), (re.compile(r'그대지'), '그 대지'),
    (re.compile(r'위건물'), '위 건물'), (re.compile(r'위토지'), '위 토지'),
    (re.compile(r'위대지'), '위 대지'),
    (re.compile(r'(?<=[가-힣])한후(?=[가-힣\s])'), '한 후'),
    (re.compile(r'(?<=[가-힣])한경우'), '한 경우'),
    (re.compile(r'(?<=[가-힣])된경우'), '된 경우'),
    (re.compile(r'(?<=[가-힣])인경우'), '인 경우'),
]

# 8. 노이즈 마커
NOISE_MARKERS = [
    re.compile(r'^[Ii]\s+[鳳凰龍]\S*\s*功?\s*', re.MULTILINE),
    re.compile(r'^m思!\s*功?\s*', re.MULTILINE),
]

In [ ]:
def is_table_line(line):
    s = line.strip()
    return (s.startswith('|') and s.endswith('|')) or (s.count('|') >= 2 and not s.startswith('#'))

def extract_frontmatter(text):
    if text.startswith('---'):
        end = text.find('---', 3)
        if end != -1:
            fm_end = end + 3
            if fm_end < len(text) and text[fm_end] == '\n': fm_end += 1
            return text[:fm_end], text[fm_end:]
    return '', text

def phase1_clean(body):
    """Phase 1: 정규식 노이즈 제거"""
    changes = 0
    # 페이지 헤더
    for p in PAGE_HEADER_PATTERNS:
        body, n = p.subn('', body); changes += n
    body, n = PAGE_COMMENT.subn('', body); changes += n
    # 가짜 헤딩
    lines = body.split('\n')
    new_lines = []
    for i, line in enumerate(lines):
        m = FALSE_HEADING.match(line.strip())
        if m:
            after = m.group(2).strip()
            if len(after) > 50:
                new_lines.append(after); changes += 1; continue
        if not is_table_line(line):
            new_line = TRAILING_PIPE.sub('', line)
            if new_line != line: changes += 1
            line = new_line
            new_line = LEADING_I.sub('', line)
            if new_line != line: changes += 1
            line = new_line
        new_lines.append(line)
    body = '\n'.join(new_lines)
    # 판례번호 + OCR
    for p, r in CASE_FIXES + OCR_FIXES:
        body, n = p.subn(r, body); changes += n
    # 노이즈 마커
    for p in NOISE_MARKERS:
        body, n = p.subn('', body); changes += n
    # 한자
    for h, k in HANJA:
        c = body.count(h); changes += c; body = body.replace(h, k)
    # 띄어쓰기
    for p, r in SPACING:
        body, n = p.subn(r, body); changes += n
    # 빈 줄 정리
    body, n = re.subn(r'\n{3,}', '\n\n', body); changes += n
    return body, changes

## 2. Phase 2 — 규칙 기반 구조화

판례 콜아웃 삽입, 볼드 키워드, 번호 리스트화 등

In [ ]:
# ═══════════════════════════════════════
# Phase 2: 규칙 기반 구조화
# ═══════════════════════════════════════

# 판례 패턴: (대판 YYYY.M.D. NNdaNNNN) 또는 (대판(전합) ...)
CASE_INLINE = re.compile(
    r'\(대판(?:\(전합\))?\s*(\d{4}\.\d{1,2}\.\d{1,2})[.,]?\s*(\d{2,4}다(?:카)?\d{2,6})\)'
)

# 번호 항목: ①②③ 등이 줄 중간에 있는 경우 → 별도 줄로
CIRCLED_NUM = re.compile(r'(?<=[.。\s])\s*([①②③④⑤⑥⑦⑧⑨⑩])\s*')

# 볼드 대상 키워드
BOLD_KEYWORDS = [
    '증명책임', '입증책임', '요건', '효과', '원칙', '예외',
    '긍정설', '부정설', '다수설', '소수설', '판례', '통설',
    '적극', '소극', '유효', '무효', '취소', '해제', '해지',
]

# 결론 키워드 (이미 볼드가 아닌 경우)
CONCLUSION_PATTERNS = [
    (re.compile(r'(?<!\*)성립O(?!\*)'), '**성립O**'),
    (re.compile(r'(?<!\*)성립X(?!\*)'), '**성립X**'),
    (re.compile(r'(?<!\*)취득O(?!\*)'), '**취득O**'),
    (re.compile(r'(?<!\*)취득X(?!\*)'), '**취득X**'),
    (re.compile(r'(?<!\*)인정O(?!\*)'), '**인정O**'),
    (re.compile(r'(?<!\*)인정X(?!\*)'), '**인정X**'),
    (re.compile(r'(?<!\*)가능O(?!\*)'), '**가능O**'),
    (re.compile(r'(?<!\*)가능X(?!\*)'), '**가능X**'),
]

# 조문번호 볼드: 제NNN조 → **제NNN조**
STATUTE_BOLD = re.compile(r'(?<!\*)(제\d{1,4}조(?:\s*제\d{1,2}항)?)(?!\*)')


def phase2_structure(body):
    """Phase 2: 규칙 기반 구조화"""
    changes = 0

    # 1. 결론 키워드 볼드
    for p, r in CONCLUSION_PATTERNS:
        body, n = p.subn(r, body); changes += n

    # 2. 조문번호 볼드 (콜아웃 내부 제외)
    lines = body.split('\n')
    new_lines = []
    in_callout = False
    for line in lines:
        if line.strip().startswith('> [!'):
            in_callout = True
        elif not line.strip().startswith('>'):
            in_callout = False

        if not in_callout and not line.strip().startswith('#'):
            new_line = STATUTE_BOLD.sub(r'**\g<1>**', line)
            if new_line != line: changes += 1
            line = new_line
        new_lines.append(line)
    body = '\n'.join(new_lines)

    # 3. 번호 항목을 별도 줄로 분리 (줄 중간의 ① 등)
    # 줄 시작이 아닌 ①②③ → 앞에 줄바꿈 + '- ' 삽입
    lines = body.split('\n')
    new_lines = []
    for line in lines:
        stripped = line.strip()
        # 이미 콜아웃이나 리스트면 스킵
        if stripped.startswith('>') or stripped.startswith('-') or stripped.startswith('#'):
            new_lines.append(line)
            continue
        # ①이 줄 시작에 있으면 리스트 아이템으로
        if stripped and stripped[0] in '①②③④⑤⑥⑦⑧⑨⑩':
            new_lines.append(f'- {stripped}')
            changes += 1
        else:
            new_lines.append(line)
    body = '\n'.join(new_lines)

    return body, changes

## 3. 통합 실행 함수

In [ ]:
def process_file(filepath, phases=(1, 2), backup=True):
    """파일 처리 (Phase 1 + Phase 2)"""
    result = {'file': filepath, 'status': 'ok', 'p1_changes': 0, 'p2_changes': 0}

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='cp949') as f:
                text = f.read()
        except:
            result['status'] = 'encoding_error'
            return result

    fm, body = extract_frontmatter(text)

    if 1 in phases:
        body, c = phase1_clean(body)
        result['p1_changes'] = c

    if 2 in phases:
        body, c = phase2_structure(body)
        result['p2_changes'] = c

    new_text = fm + body
    if new_text == text:
        result['status'] = 'unchanged'
        return result

    if backup:
        bak_dir = os.path.join(os.path.dirname(filepath), BACKUP_SUFFIX)
        os.makedirs(bak_dir, exist_ok=True)
        bak_path = os.path.join(bak_dir, os.path.basename(filepath))
        if not os.path.exists(bak_path):  # 최초 백업만
            shutil.copy2(filepath, bak_path)

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(new_text)
    result['status'] = 'applied'
    return result


def collect_files(target_dir):
    """대상 .md 파일 수집"""
    files = []
    for root, dirs, fnames in os.walk(target_dir):
        dirs[:] = [d for d in dirs if not d.startswith('_')]
        for fn in fnames:
            if fn.endswith('.md') and not fn.startswith('_'):
                files.append(os.path.join(root, fn))
    return sorted(files)

## 4. 배치 실행 — 전체 교재원문 (Phase 1 + 2)

In [ ]:
# 대상 선택: 전체 또는 특정 교재
# 전체: TEXTBOOK_DIR
# 특정 교재: os.path.join(TEXTBOOK_DIR, '민법/송영곤_쟁점노트')

TARGET = TEXTBOOK_DIR  # 전체 실행
# TARGET = os.path.join(TEXTBOOK_DIR, '민법/송영곤_쟁점노트')  # 특정 교재

files = collect_files(TARGET)
print(f'대상 파일 수: {len(files)}')

# 과목별 분포
subj_count = Counter()
for f in files:
    parts = f.replace(TEXTBOOK_DIR, '').split('/')
    if len(parts) >= 2:
        subj_count[parts[1]] += 1
print('과목별:', dict(subj_count))

In [ ]:
# Phase 1 + 2 배치 실행
results = []
stats = Counter()
total_p1 = 0
total_p2 = 0

for i, fp in enumerate(files):
    r = process_file(fp, phases=(1, 2), backup=BACKUP_ENABLED)
    results.append(r)
    stats[r['status']] += 1
    total_p1 += r.get('p1_changes', 0)
    total_p2 += r.get('p2_changes', 0)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{len(files)}] P1={total_p1} P2={total_p2}')

print(f'\n완료: {len(files)}파일')
print(f'상태: {dict(stats)}')
print(f'Phase 1 변경: {total_p1}')
print(f'Phase 2 변경: {total_p2}')

In [ ]:
# 리포트 저장
report = {
    'timestamp': datetime.now().isoformat(),
    'target': TARGET,
    'total_files': len(files),
    'stats': dict(stats),
    'total_p1_changes': total_p1,
    'total_p2_changes': total_p2,
    'errors': [r for r in results if r['status'] not in ('applied', 'unchanged')],
}
report_path = os.path.join(REPORT_DIR, f'colab_batch_{datetime.now():%Y%m%d_%H%M}.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f'리포트: {report_path}')

## 5. Phase 3 — Claude Code에서 처리

> Phase 3(LLM 패스)는 이 노트북이 아니라 **Claude Code 세션**에서 에이전트 병렬로 처리합니다.
> 
> Colab에서 Phase 1+2 완료 후 → Claude Code에서 교재 단위로 에이전트 3~5개 병렬 실행
>
> **작업 내용**: 헤딩 위계 정규화, 판례 콜아웃(`[!판례]`), 학설 표(`[!학설]`), 사실관계 콜아웃, 논리적 줄바꿈, 끊긴 문장 재접합
>
> 아래 셀은 Phase 3 없이 바로 검증(섹션 6)으로 넘어가세요.

In [ ]:
# Phase 3은 Claude Code에서 처리합니다.
# 이 셀과 다음 셀은 실행하지 마세요.
# 바로 섹션 6(검증)으로 넘어가세요.
print("Phase 3 → Claude Code 에이전트에서 처리. 이 셀 건너뛰세요.")

In [ ]:
# (참고용) Phase 3 프롬프트 — Claude Code 에이전트에서 사용하는 리플로우 규칙
# Gemini API 키가 있다면 이 셀을 활성화하여 Colab에서도 Phase 3 실행 가능

REFLOW_PROMPT = """
[Phase 3 프롬프트는 Claude Code .agent/scripts/reflow_batch.py 및 
.agent/skills/textbook-reflow/SKILL.md 참조]

커스텀 콜아웃 타입:
- [!조문]: 보라 — 조문 원문
- [!판례]: 금색 — 판례 판시요지  
- [!판례변경]: 빨강 — 판례 변경/폐기
- [!학설]: 파랑 — 학설 대립
- [!답안]: 청록 — 시험 답안 문구
- [!함정]: 주황 — 시험 함정/주의
- [!사실관계]: 회녹 — 사례 사실관계

CSS: sync/.obsidian/snippets/law-callouts.css
"""
print("Phase 3 프롬프트 참조용. 실행 불필요.")

In [ ]:
# Phase 3 배치 — 스킵 (Claude Code에서 처리)
print("Phase 3 → Claude Code 에이전트에서 병렬 처리. 섹션 6으로 이동하세요.")

## 6. 검증

리플로우 전후 조문/판례 카운트 비교

In [ ]:
def count_references(filepath):
    """파일 내 조문/판례 참조 개수"""
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    statutes = len(re.findall(r'제\d{1,4}조', text))
    cases = len(re.findall(r'\d{2,4}다(?:카)?\d{2,6}', text))
    return statutes, cases

# 백업 vs 현재 비교
verify_target = os.path.join(TEXTBOOK_DIR, '민법/송영곤_쟁점노트')
verify_files = collect_files(verify_target)

mismatches = []
for fp in verify_files:
    bak = os.path.join(os.path.dirname(fp), BACKUP_SUFFIX, os.path.basename(fp))
    if not os.path.exists(bak):
        continue
    s_old, c_old = count_references(bak)
    s_new, c_new = count_references(fp)
    if c_new < c_old:
        mismatches.append({
            'file': os.path.basename(fp),
            'cases_before': c_old, 'cases_after': c_new,
            'lost': c_old - c_new
        })

if mismatches:
    print(f'⚠️ 판례번호 누락 의심: {len(mismatches)}파일')
    for m in mismatches:
        print(f'  {m["file"]}: {m["cases_before"]}→{m["cases_after"]} (누락 {m["lost"]}건)')
else:
    print('✅ 판례번호 누락 없음')

## 7. 조문/판례 추출 (Phase 4-5)

전체 파일에서 조문/판례 참조 추출 → JSON 인덱스 생성

In [ ]:
CASE_PATTERN = re.compile(
    r'대판(?:\(전합\))?\s*(\d{4}\.\d{1,2}\.\d{1,2})[.,]?\s*(\d{2,4}다(?:카)?\d{2,6})'
)
STATUTE_PATTERN = re.compile(r'제(\d{1,4})조')

all_cases = Counter()
all_statutes = Counter()
case_details = {}  # number → {date, files}

for fp in files:
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    fname = os.path.basename(fp)

    for m in CASE_PATTERN.finditer(text):
        num = m.group(2)
        all_cases[num] += 1
        if num not in case_details:
            case_details[num] = {'date': m.group(1), 'files': []}
        if fname not in case_details[num]['files']:
            case_details[num]['files'].append(fname)

    for m in STATUTE_PATTERN.finditer(text):
        all_statutes[f'제{m.group(1)}조'] += 1

print(f'고유 판례: {len(all_cases)}개')
print(f'고유 조문: {len(all_statutes)}개')
print(f'\n판례 상위 20:')
for num, cnt in all_cases.most_common(20):
    d = case_details[num]
    print(f'  {num} ({d["date"]}): {cnt}회 / {len(d["files"])}파일')

In [ ]:
# 판례 빈도 JSON 저장
case_index = {
    'timestamp': datetime.now().isoformat(),
    'total_unique': len(all_cases),
    'cases': {
        num: {
            'count': cnt,
            'date': case_details[num]['date'],
            'files': case_details[num]['files'],
        }
        for num, cnt in all_cases.most_common()
    }
}
idx_path = os.path.join(REPORT_DIR, '판례빈도_전체.json')
with open(idx_path, 'w', encoding='utf-8') as f:
    json.dump(case_index, f, ensure_ascii=False, indent=2)
print(f'저장: {idx_path}')